# PyOxigraph 入門

- [pyoxigraph 0.5.11 documentation](https://pyoxigraph.readthedocs.io/en/stable/)

## 最小例

In [1]:
from pyoxigraph import Store, NamedNode, Literal, Quad, QuerySolutions

store = Store()

ex = NamedNode("http://example/")
schema_name = NamedNode("http://schema.org/name")

store.add(
    Quad(ex, schema_name, Literal("example"))
)

# SPARQL クエリ
results = store.query(
    """
    SELECT ?name
    WHERE {
        <http://example/> <http://schema.org/name> ?name
    }
    """
)

assert isinstance(results, QuerySolutions)
for binding in results:
    print(binding["name"])

"example"


## Named Graph を保存

In [2]:
store = Store()

wallet = NamedNode("https://example.org/wallet")
located_in = NamedNode("https://example.org/locatedIn")
kitchen = NamedNode("https://example.org/kitchen")
observed = NamedNode("https://example.org/observed")

store.add(Quad(wallet, located_in, kitchen, observed))

print("=== Quads ===")
for quad in store:
    print(quad)

print("\n=== Named Graphs ===")
for graph_name in store.named_graphs():
    print(graph_name)

=== Quads ===
<https://example.org/wallet> <https://example.org/locatedIn> <https://example.org/kitchen> <https://example.org/observed>

=== Named Graphs ===
<https://example.org/observed>


## Python で Quad を検索

In [3]:
matches = store.quads_for_pattern(
    wallet,  # subject
    located_in,  # predicate
    None,  # object
    observed,  # graph name
)

for quad in matches:
    print("Location:", quad.object)

Location: <https://example.org/kitchen>


## 永続化

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
import gc

with TemporaryDirectory() as tmpdir:
    db_path = Path(tmpdir) / "rdf_store"

    # ① ディスク上の Store を開く
    first = Store(db_path)

    first.add(Quad(wallet, located_in, kitchen, observed))

    first.flush()

    # ② 一度閉じる
    del first
    gc.collect()

    # ③ 同じディレクトリから開き直す
    second = Store(db_path)

    restored = list(second.quads_for_pattern(wallet, located_in, None, observed))

    print("Restored quads:", len(restored))
    print("Restored location:", restored[0].object)

    del second
    gc.collect()


Restored quads: 1
Restored location: <https://example.org/kitchen>


## RDFLib との相互運用

### RDFLib → PyOxigraph

In [5]:
from rdflib import Dataset, Namespace

EX = Namespace("https://example.org/")

source_ds = Dataset()
source_ds.bind("ex", EX)

source_ds.graph(EX.observed).add((EX.wallet, EX.onTopOf, EX.table))

source_ds.graph(EX.inferred).add((EX.wallet, EX.locatedIn, EX.kitchen))

# 一旦 Trig としてシリアライズ
trig_text: str = source_ds.serialize(format="trig")

print(trig_text)

@prefix ex: <https://example.org/> .

ex:observed {
    ex:wallet ex:onTopOf ex:table .
}

ex:inferred {
    ex:wallet ex:locatedIn ex:kitchen .
}




In [6]:
from pyoxigraph import Store, RdfFormat

ox_store = Store()

# Trig 形式をロード
ox_store.load(
    input=trig_text,
    format=RdfFormat.TRIG,
)

print("Quad count:", len(ox_store))

print("\nNamed Graphs:")
for graph_name in ox_store.named_graphs():
    print(graph_name.value)

Quad count: 2

Named Graphs:
https://example.org/observed
https://example.org/inferred


### PyOxigraph → RDFLib

In [7]:
# PyOxigraph → TriG（bytes）
trig_bytes = ox_store.dump(format=RdfFormat.TRIG)
assert trig_bytes is not None

In [8]:
trig_bytes

b'<https://example.org/inferred> {\n\t<https://example.org/wallet> <https://example.org/locatedIn> <https://example.org/kitchen> .\n}\n<https://example.org/observed> {\n\t<https://example.org/wallet> <https://example.org/onTopOf> <https://example.org/table> .\n}\n'

In [9]:
# TriG → RDFLib
assert trig_bytes is not None
restored_ds = Dataset()
restored_ds.parse(
    data=trig_bytes.decode("utf-8"),
    format="trig",
)

# Quad 集合として一致を検証
original = set(source_ds.quads((None, None, None, None)))

restored = set(restored_ds.quads((None, None, None, None)))

print("Same RDF Dataset:", original == restored)


Same RDF Dataset: True


## RDF を更新

In [10]:
from pyoxigraph import NamedNode, Quad

wallet = NamedNode(str(EX.wallet))
located_in = NamedNode(str(EX.locatedIn))

inferred = NamedNode(str(EX.inferred))
kitchen = NamedNode(str(EX.kitchen))
living_room = NamedNode(str(EX.livingRoom))

old_quad = Quad(wallet, located_in, kitchen, inferred)

new_quad = Quad(wallet, located_in, living_room, inferred)


for quad in ox_store.quads_for_pattern(wallet, located_in, None, inferred):
    print("Location:", quad.object)

ox_store.remove(old_quad)
ox_store.add(new_quad)

for quad in ox_store.quads_for_pattern(wallet, located_in, None, inferred):
    print("Location:", quad.object)

Location: <https://example.org/kitchen>
Location: <https://example.org/livingRoom>


## SPARQL Update

In [11]:
ox_store.update("""
PREFIX ex: <https://example.org/>

DELETE {
    GRAPH ex:inferred {
        ex:wallet ex:locatedIn ex:livingRoom .
    }
}
INSERT {
    GRAPH ex:inferred {
        ex:wallet ex:locatedIn ex:bedroom .
    }
}
WHERE {
    GRAPH ex:inferred {
        ex:wallet ex:locatedIn ex:livingRoom .
    }
}
""")

for quad in ox_store.quads_for_pattern(wallet, located_in, None, inferred):
    print("Location:", quad.object)

Location: <https://example.org/bedroom>


## oxrdflib

[oxigraph/oxrdflib: Oxrdflib provides rdflib stores using pyoxigraph](https://github.com/oxigraph/oxrdflib)  
注：oxrdflib は RDFLib のすべての機能と完全互換ではない

In [12]:
from rdflib import Dataset

# 通常の RDFLib Dataset
memory_ds = Dataset()

# Oxigraph を Store として使う Dataset
ox_ds = Dataset(store="Oxigraph")

ox_ds.graph(EX.observed).add((EX.wallet, EX.onTopOf, EX.table))

print(list(ox_ds.graph(EX.observed).triples((EX.wallet, None, None))))

ox_ds.close()

[(rdflib.term.URIRef('https://example.org/wallet'), rdflib.term.URIRef('https://example.org/onTopOf'), rdflib.term.URIRef('https://example.org/table'))]
